# Movie Recommender System

Interactive explorer for the [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/) dataset (9,742 movies).  
Use the controls in each section to search and filter movies in real time.

In [ ]:
%matplotlib inline
from modules.recommender import Recommender
import ipywidgets as widgets
from IPython.display import display

recommender = Recommender(
    'data/ml-latest-small/movies.csv',
    'data/ml-latest-small/ratings.csv'
)

# Pre-compute lookup values — avoid __getattr__ calls for 9k movies
genres_sorted  = sorted(recommender.get_all_genres())
year_min       = min(m.year for m in recommender.movies if m.year > 0)
year_max       = max(m.year for m in recommender.movies if m.year > 0)
decades_sorted = sorted(set(f"{(m.year // 10) * 10}s" for m in recommender.movies if m.year > 0))

print(f'Loaded {len(recommender.movies)} movies  |  '
      f'{len(genres_sorted)} genres  |  '
      f'Years {year_min}\u2013{year_max}')

---
## Interactive Movie Explorer

In [ ]:
# ── Search by Title ───────────────────────────────────────────────────────────
title_input = widgets.Text(
    placeholder='e.g. Star Wars, Batman, Toy…',
    description='Title:',
    layout=widgets.Layout(width='380px')
)
title_topn = widgets.IntSlider(
    value=10, min=1, max=30,
    description='Max results:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)
title_out = widgets.Output()

def _search_title(change):
    title_out.clear_output(wait=True)
    with title_out:
        query = title_input.value.strip()
        if not query:
            print('Type a title fragment to search.')
            return
        hits = [m for m in recommender.movies if query.lower() in m.title.lower()]
        hits.sort(key=lambda m: m.rating, reverse=True)
        shown = hits[:title_topn.value]
        print(f'{len(hits)} match(es) — showing top {len(shown)}:\n')
        for i, m in enumerate(shown, 1):
            genres = ', '.join(m.genre) if m.genre else '—'
            print(f'  {i:>2}. {m.title} ({m.year})  |  {genres}  |  ★ {m.rating:.2f}')

title_input.observe(_search_title, names='value')
title_topn.observe(_search_title, names='value')

display(widgets.VBox([
    widgets.HTML('<b style="font-size:1.05em">Search by Title</b>'),
    widgets.HBox([title_input, title_topn]),
    title_out
]))

In [ ]:
# ── Filter by Genre ───────────────────────────────────────────────────────────
genre_dd = widgets.Dropdown(
    options=genres_sorted, value='Action', description='Genre:'
)
genre_topn = widgets.IntSlider(
    value=10, min=1, max=30,
    description='Top N:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
genre_out = widgets.Output()

def _filter_genre(change):
    genre_out.clear_output(wait=True)
    with genre_out:
        results = recommender.recommend_by_genre(genre_dd.value, genre_topn.value)
        print(f'Top {len(results)} {genre_dd.value} movies:\n')
        for i, m in enumerate(results, 1):
            print(f'  {i:>2}. {m.title} ({m.year})  |  {", ".join(m.genre)}  |  ★ {m.rating:.2f}')

genre_dd.observe(_filter_genre, names='value')
genre_topn.observe(_filter_genre, names='value')

with genre_out:
    print('Select a genre above to see results.')

display(widgets.VBox([
    widgets.HTML('<b style="font-size:1.05em">Filter by Genre</b>'),
    widgets.HBox([genre_dd, genre_topn]),
    genre_out
]))

In [ ]:
# ── Filter by Release Year ────────────────────────────────────────────────────
year_slider = widgets.IntSlider(
    value=2000, min=year_min, max=year_max, step=1,
    description='Year:',
    continuous_update=False,
    layout=widgets.Layout(width='420px')
)
year_topn = widgets.IntSlider(
    value=10, min=1, max=30,
    description='Top N:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px')
)
year_out = widgets.Output()

def _filter_year(change):
    year_out.clear_output(wait=True)
    with year_out:
        results = recommender.recommend_by_year(year_slider.value, year_topn.value)
        if not results:
            print(f'No movies found for {year_slider.value}.')
            return
        print(f'Top {len(results)} movies from {year_slider.value}:\n')
        for i, m in enumerate(results, 1):
            print(f'  {i:>2}. {m.title}  |  {", ".join(m.genre)}  |  ★ {m.rating:.2f}')

year_slider.observe(_filter_year, names='value')
year_topn.observe(_filter_year, names='value')

with year_out:
    print('Move the slider to pick a year.')

display(widgets.VBox([
    widgets.HTML('<b style="font-size:1.05em">Filter by Release Year</b>'),
    widgets.HBox([year_slider, year_topn]),
    year_out
]))

In [ ]:
# ── Filter by Decade ──────────────────────────────────────────────────────────
decade_dd = widgets.Dropdown(
    options=decades_sorted, value='2000s', description='Decade:'
)
decade_topn = widgets.IntSlider(
    value=10, min=1, max=30,
    description='Top N:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)
decade_out = widgets.Output()

def _filter_decade(change):
    decade_out.clear_output(wait=True)
    with decade_out:
        results = recommender.recommend_by_decade(decade_dd.value, decade_topn.value)
        print(f'Top {len(results)} movies from the {decade_dd.value}:\n')
        for i, m in enumerate(results, 1):
            print(f'  {i:>2}. {m.title} ({m.year})  |  {", ".join(m.genre)}  |  ★ {m.rating:.2f}')

decade_dd.observe(_filter_decade, names='value')
decade_topn.observe(_filter_decade, names='value')

with decade_out:
    print('Select a decade above to see results.')

display(widgets.VBox([
    widgets.HTML('<b style="font-size:1.05em">Filter by Decade</b>'),
    widgets.HBox([decade_dd, decade_topn]),
    decade_out
]))

In [ ]:
# ── Filter by Two Genres (intersection) ──────────────────────────────────────
genre1_dd = widgets.Dropdown(
    options=genres_sorted, value='Action', description='Genre 1:'
)
genre2_dd = widgets.Dropdown(
    options=genres_sorted, value='Drama', description='Genre 2:'
)
two_genre_topn = widgets.IntSlider(
    value=10, min=1, max=30,
    description='Top N:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px')
)
two_genre_out = widgets.Output()

def _filter_two_genres(change):
    two_genre_out.clear_output(wait=True)
    with two_genre_out:
        if genre1_dd.value == genre2_dd.value:
            print('Select two different genres.')
            return
        results = recommender.recommend_by_multiple_genres(
            genre1_dd.value, genre2_dd.value, two_genre_topn.value
        )
        label = f'{genre1_dd.value} ∩ {genre2_dd.value}'
        if not results:
            print(f'No movies found in both {label}.')
            return
        print(f'Top {len(results)} movies in {label}:\n')
        for i, m in enumerate(results, 1):
            print(f'  {i:>2}. {m.title} ({m.year})  |  {", ".join(m.genre)}  |  ★ {m.rating:.2f}')

genre1_dd.observe(_filter_two_genres, names='value')
genre2_dd.observe(_filter_two_genres, names='value')
two_genre_topn.observe(_filter_two_genres, names='value')

with two_genre_out:
    print('Select two genres above to see their intersection.')

display(widgets.VBox([
    widgets.HTML('<b style="font-size:1.05em">Filter by Two Genres (intersection)</b>'),
    widgets.HBox([genre1_dd, genre2_dd, two_genre_topn]),
    two_genre_out
]))

In [ ]:
# ── Filter by Minimum Rating ──────────────────────────────────────────────────
rating_slider = widgets.FloatSlider(
    value=4.0, min=0.0, max=5.0, step=0.1,
    description='Min rating:',
    continuous_update=False,
    readout_format='.1f',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
rating_topn = widgets.IntSlider(
    value=15, min=1, max=50,
    description='Max results:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)
rating_out = widgets.Output()

def _filter_rating(change):
    rating_out.clear_output(wait=True)
    with rating_out:
        results = recommender.get_rated_above(rating_slider.value)
        results.sort(key=lambda m: m.rating, reverse=True)
        shown = results[:rating_topn.value]
        print(f'{len(results)} movies rated ≥ {rating_slider.value:.1f} — showing {len(shown)}:\n')
        for i, m in enumerate(shown, 1):
            genres = ', '.join(m.genre) if m.genre else '—'
            print(f'  {i:>2}. {m.title} ({m.year})  |  {genres}  |  ★ {m.rating:.2f}')

rating_slider.observe(_filter_rating, names='value')
rating_topn.observe(_filter_rating, names='value')

with rating_out:
    print('Move the slider to set a minimum rating.')

display(widgets.VBox([
    widgets.HTML('<b style="font-size:1.05em">Filter by Minimum Rating</b>'),
    widgets.HBox([rating_slider, rating_topn]),
    rating_out
]))

---
## Statistics & Visualization

In [ ]:
# Rating statistics computed with numpy
stats = recommender.get_rating_stats()
print('Rating statistics across all movies:')
print(f"  Mean:   {stats['mean']:.3f}")
print(f"  Median: {stats['median']:.3f}")
print(f"  Std:    {stats['std']:.3f}")
print(f"  Min:    {stats['min']:.3f}")
print(f"  Max:    {stats['max']:.3f}")

In [ ]:
# Rating distribution histogram (matplotlib)
import matplotlib.pyplot as plt
import numpy as np

ratings = [m.rating for m in recommender.movies if m.rating > 0]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ratings, bins=20, range=(0, 5), edgecolor='black', color='steelblue', alpha=0.85)
ax.axvline(np.mean(ratings), color='crimson', linestyle='--', linewidth=1.4, label=f'Mean {np.mean(ratings):.2f}')
ax.axvline(np.median(ratings), color='darkorange', linestyle=':', linewidth=1.4, label=f'Median {np.median(ratings):.2f}')
ax.set_title('Movie Rating Distribution (movies with at least one rating)')
ax.set_xlabel('Average Rating')
ax.set_ylabel('Number of Movies')
ax.legend()
plt.tight_layout()
plt.show()

---
## Feature Demonstrations

The cells below demonstrate each required Python feature used in the implementation.

In [ ]:
# __getattr__ — decade and is_classic are computed dynamically from Movie.year
for movie in recommender.recommend_top_rated(5):
    print(f"{movie.title} ({movie.year})  decade={movie.decade}  classic={movie.is_classic}")

In [ ]:
# Generator — stream_movies() yields one Movie at a time
gen = recommender.stream_movies()
print('First 5 via generator:')
for i, movie in enumerate(gen, 1):
    print(f'  {i}. {movie.title}')
    if i == 5:
        break

In [ ]:
# Set operations — get_all_genres uses a set comprehension; recommend_by_multiple_genres uses &
all_genres = recommender.get_all_genres()
print(f'All unique genres ({len(all_genres)} total):', sorted(all_genres))

In [ ]:
# Recursion — _recursive_top_n pulls the max each call, then recurses on the remainder
print('Top 5 via recursion:')
for i, m in enumerate(recommender._recursive_top_n(recommender.movies, 5), 1):
    print(f'  {i}. {m.title} ({m.rating})')

In [ ]:
# Built-in libs — random.sample and itertools.combinations
print('5 random picks:')
for m in recommender.recommend_random(5):
    print(f'  {m.title} ({m.rating:.2f})')

print('\nPairs from top 4 (itertools.combinations):')
for m1, m2 in recommender.get_movie_pairs(top_n=4):
    print(f'  {m1.title}  \u2194  {m2.title}')

In [ ]:
# map / filter / zip / reduce
titles = recommender.get_titles()
print(f'map  \u2014 first 5 titles: {titles[:5]}')

highly = recommender.get_rated_above(4.5)
print(f'filter \u2014 movies rated \u2265 4.5: {len(highly)}')

pairs = recommender.get_title_rating_pairs()
print(f'zip  \u2014 first pair: {pairs[0]}')

total = recommender.get_total_rating()
print(f'reduce \u2014 total rating sum: {total:.1f}, average: {total / len(recommender.movies):.3f}')

In [ ]:
# Dict comprehension — {genre: count}
genre_counts = recommender.get_genre_counts()
print('Movies per genre (top 10 by count):')
for genre, count in sorted(genre_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f'  {genre}: {count}')